In [1]:
import os
import sys
import sqlite3
from pathlib import Path
import importlib.util

# ===== 共通ユーティリティ =====
def find_config_py(start: Path, max_up: int = 6) -> Path | None:
    """start から最大 max_up 階層まで上に遡って utils/config.py を探す"""
    cur = start.resolve()
    for _ in range(max_up + 1):
        candidate = cur / "utils" / "config.py"
        if candidate.is_file():
            return candidate
        cur = cur.parent
    return None

def import_config_from_path(cfg_path: Path):
    """ファイルパスから utils.config を読み込む（パッケージでなくてもOK）"""
    spec = importlib.util.spec_from_file_location("utils.config", cfg_path)
    if spec and spec.loader:
        module = importlib.util.module_from_spec(spec)
        sys.modules["utils.config"] = module
        spec.loader.exec_module(module)
        return module
    raise ImportError(f"Failed to import config from: {cfg_path}")

# ===== ベースディレクトリ決定（スクリプト/ノートブック両対応） =====
try:
    base_dir = Path(__file__).resolve().parent
except NameError:
    base_dir = Path.cwd()

# ===== config の解決 =====
config_module = None

# 1) 近傍から utils/config.py を探索して直接 import
cfg_path = find_config_py(base_dir)
if cfg_path:
    config_module = import_config_from_path(cfg_path)
else:
    # 2) 親ディレクトリを sys.path に追加して通常 import を試行
    sys.path.append(str((base_dir / "..").resolve()))
    try:
        from utils.config import PROJECT_DIR  # type: ignore
    except Exception:
        PROJECT_DIR = None
    else:
        config_module = sys.modules.get("utils.config")

# 3) フォールバック: 環境変数 or カレント名
if config_module is None:
    PROJECT_DIR = os.getenv("PROJECT_DIR", base_dir.name)
else:
    PROJECT_DIR = getattr(config_module, "PROJECT_DIR", os.getenv("PROJECT_DIR", base_dir.name))

# ===== パス定義 =====
if os.name == "nt":
    home = Path(os.environ.get("USERPROFILE", str(Path.home())))
else:
    home = Path.home()

user_base = home / "myenv310" / PROJECT_DIR
db_dir = user_base / "db"
db_dir.mkdir(parents=True, exist_ok=True)

db_path = db_dir / "output.db"
columns_file = db_dir / "columns_with_type.txt"

# 予備: もし上の columns_file が無い場合は、プロジェクトルート直下/実行場所直下も探す
if not columns_file.exists():
    # 近傍の utils/config.py が見つかったなら、その親をプロジェクトルート候補に
    project_root = cfg_path.parent.parent if cfg_path else base_dir
    alt_candidates = [
        project_root / "db" / "columns_with_type.txt",
        base_dir / "db" / "columns_with_type.txt",
        Path.cwd() / "db" / "columns_with_type.txt",
    ]
    for c in alt_candidates:
        if c.exists():
            columns_file = c
            break

# ===== カラム+型 読み込み =====
if not columns_file.exists():
    raise FileNotFoundError(f"columns_with_type.txt が見つかりません: {columns_file}")

with columns_file.open(encoding="utf-8") as f:
    cols_types = [line.strip() for line in f if line.strip()]

if not cols_types:
    raise ValueError("カラム設定ファイルが空です")

columns_definitions = ",\n    ".join(cols_types)

table_name = "result_table"

# ===== CREATE TABLE 文 =====
create_table_sql = f"""
DROP TABLE IF EXISTS {table_name};
CREATE TABLE {table_name} (
    {columns_definitions}
);
"""

print("[INFO] ✅ SQL定義:")
print(create_table_sql)

# ===== DB作成・テーブル作成 =====
conn = sqlite3.connect(str(db_path))
try:
    cur = conn.cursor()
    cur.executescript(create_table_sql)
    conn.commit()
finally:
    conn.close()

print(f"[INFO] ✅ SQLite DB作成完了: {db_path}")
print(f"[INFO] PROJECT_DIR = {PROJECT_DIR}")
print(f"[INFO] columns_file = {columns_file}")

[INFO] ✅ SQL定義:

DROP TABLE IF EXISTS result_table;
CREATE TABLE result_table (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT UNIQUE,
    shop_name TEXT,
    shop_product_id TEXT,
    scraped_date TEXT,
    created_at TEXT,
    updated_at TEXT,
    machine_name TEXT,
    normalized_machine_name TEXT,
    price INTEGER,
    product_url TEXT,
    image_url TEXT,
    stock TEXT,
    maker TEXT,
    master_machine_id TEXT,
    master_machine_pworld_url TEXT,
    master_machine_pworldimage_url TEXT,
    master_machine_name TEXT,
    master_machine_model_TEXT,
    master_machine_maker TEXT,
    master_machine_type TEXT,
    master_machine_gouki TEXT,
    master_machine_memo TEXT,
    master_machine_tag TEXT,
    master_machine_dis TEXT,
    status TEXT,
    remarks TEXT
);

[INFO] ✅ SQLite DB作成完了: C:\Users\Owner\myenv310\soubanavi-s\db\output.db
[INFO] PROJECT_DIR = soubanavi-s
[INFO] columns_file = C:\Users\Owner\myenv310\soubanavi-s\db\columns_with_type.txt
